# Join market data, Google Trends, and engineered features

This notebook:

1. Loads **technical indicators** from `crypto_prices_with_indicators.csv`
2. Adds **macro + news sentiment** columns from `crypto_prices_macro_sentiment.csv` (merged on `date`)
3. Merges **Google Trends** (`google_trends_clean.csv`) on `date`
4. Builds **ML-ready composite features** (momentum gaps, macro stress, trend z-scores, etc.)
5. Exports `eth_ml_dataset.csv` and `eth_ml_dataset_features.csv` (same table for downstream scripts)

In [57]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

INDICATORS_PATH = "crypto_prices_with_indicators.csv"
MACRO_SENTIMENT_PATH = "crypto_prices_macro_sentiment.csv"
TRENDS_PATH = "google_trends_clean.csv"
OUTPUT_PATH = "eth_ml_dataset.csv"
OUTPUT_FEATURES_PATH = "eth_ml_dataset_features.csv"

In [58]:
def _standardize_date_col(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" not in out.columns and "Date" in out.columns:
        out = out.rename(columns={"Date": "date"})
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out.dropna(subset=["date"]).sort_values("date").drop_duplicates(subset=["date"], keep="last")


# 1) Technical indicators + base OHLCV-derived columns
prices = pd.read_csv(INDICATORS_PATH)
prices = _standardize_date_col(prices)

# 2) Macro + news sentiment (columns not already in indicators file)
macro = pd.read_csv(MACRO_SENTIMENT_PATH)
macro = _standardize_date_col(macro)
extra_cols = [c for c in macro.columns if c != "date" and c not in prices.columns]
if extra_cols:
    prices = prices.merge(macro[["date"] + extra_cols], on="date", how="left")

# 3) Google Trends
trends = pd.read_csv(TRENDS_PATH)
trends = _standardize_date_col(trends)

print("Prices (+macro) shape:", prices.shape)
print("Trends shape:", trends.shape)
prices.head()

Prices (+macro) shape: (2083, 86)
Trends shape: (3026, 16)


,date,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,eth_ema_12,eth_ema_26,eth_sma_ratio_10,eth_ema_ratio_12_26,eth_vol_7d,eth_vol_14d,eth_bb_z_20,eth_rsi_14,eth_macd,eth_macd_signal,eth_macd_hist,eth_obv,btc_ret_1d,btc_ret_3d,btc_ret_7d,btc_sma_10,btc_sma_30,btc_ema_12,btc_ema_26,btc_sma_ratio_10,btc_ema_ratio_12_26,btc_vol_7d,btc_vol_14d,btc_bb_z_20,btc_rsi_14,btc_macd,btc_macd_signal,btc_macd_hist,btc_obv,vix_ret_1d,vix_ret_3d,vix_ret_7d,vix_sma_10,vix_sma_30,vix_ema_12,vix_ema_26,vix_sma_ratio_10,vix_ema_ratio_12_26,vix_vol_7d,vix_vol_14d,vix_bb_z_20,vix_rsi_14,vix_macd,vix_macd_signal,vix_macd_hist,vix_obv,gc=f_ret_1d,gc=f_ret_3d,gc=f_ret_7d,gc=f_sma_10,gc=f_sma_30,gc=f_ema_12,gc=f_ema_26,gc=f_sma_ratio_10,gc=f_ema_ratio_12_26,gc=f_vol_7d,gc=f_vol_14d,gc=f_bb_z_20,gc=f_rsi_14,gc=f_macd,gc=f_macd_signal,gc=f_macd_hist,gc=f_obv,btc_eth_spread_1d,eth_news_sent_mean,eth_news_sent_wmean,eth_news_sent_std,eth_news_count,eth_news_pos_ratio,eth_news_neg_ratio,federal_funds_rate,cpi,treasury_10y,fed_rate_change_7d,treasury_10y_change_7d,cpi_yoy_pct
0,2018-01-02,1313.699951,9.77,884.443970,14982.099609,NaN,NaN,NaN,NaN,NaN,884.443970,884.443970,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,14982.099609,14982.099609,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,9.770000,9.770000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,1313.699951,1313.699951,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.46,NaN,NaN,NaN
1,2018-01-03,1316.199951,9.15,962.719971,15201.000000,0.088503,NaN,NaN,NaN,NaN,896.486431,890.242192,NaN,0.007014,NaN,NaN,NaN,NaN,6.244239,1.248848,4.995392,5.093160e+09,0.014611,NaN,NaN,NaN,NaN,15015.776593,14998.314453,NaN,0.001164,NaN,NaN,NaN,NaN,17.462139,3.492428,13.969712,1.687190e+10,-0.063460,NaN,NaN,NaN,NaN,9.674616,9.724074,NaN,-0.005086,NaN,NaN,NaN,NaN,-0.049459,-0.009892,-0.039567,0.0,0.001903,NaN,NaN,NaN,NaN,1314.084567,1313.885136,NaN,0.000152,NaN,NaN,NaN,NaN,0.199430,0.039886,0.159544,42.0,-0.073892,0.03173,0.03173,0.0,1.0,0.0,0.0,1.42,247.867,2.44,NaN,NaN,NaN
2,2018-01-04,1319.400024,9.22,980.921997,15599.200195,0.018907,NaN,NaN,NaN,NaN,909.476518,896.959215,NaN,0.013955,NaN,NaN,NaN,NaN,12.517304,3.502539,9.014765,1.159602e+10,0.026196,NaN,NaN,NaN,NaN,15105.534070,15042.824508,NaN,0.004169,NaN,NaN,NaN,NaN,62.709562,15.335855,47.373707,3.865510e+10,0.007650,NaN,NaN,NaN,NaN,9.604675,9.686736,NaN,-0.008471,NaN,NaN,NaN,NaN,-0.082061,-0.024326,-0.057735,0.0,0.002431,NaN,NaN,NaN,NaN,1314.902329,1314.293647,NaN,0.000463,NaN,NaN,NaN,NaN,0.608683,0.153645,0.455037,44.0,0.007289,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.46,NaN,NaN,NaN
3,2018-01-05,1320.300049,9.22,997.719971,17429.500000,0.017125,0.128076,NaN,NaN,NaN,923.052434,904.422974,NaN,0.020598,NaN,NaN,NaN,NaN,18.629460,6.527923,12.101537,1.827917e+10,0.117333,0.163355,NaN,NaN,NaN,15463.067290,15219.615285,NaN,0.015996,NaN,NaN,NaN,NaN,243.452005,60.959085,182.492920,6.249600e+10,0.000000,-0.056295,NaN,NaN,NaN,9.545494,9.652163,NaN,-0.011051,NaN,NaN,NaN,NaN,-0.106668,-0.040794,-0.065874,0.0,0.000682,0.005024,NaN,NaN,NaN,1315.732748,1314.738565,NaN,0.000756,NaN,NaN,NaN,NaN,0.994182,0.321753,0.672430,45.0,0.100208,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.47,NaN,NaN,NaN
4,2018-01-08,1318.599976,9.52,1148.530029,15170.099609,0.151155,0.193005,NaN,NaN,NaN,957.741295,922.504978,NaN,0.038196,NaN,NaN,NaN,NaN,35.236317,12.269602,22.966715,2.673014e+10,-0.129631,-0.002033,NaN,NaN,NaN,15417.995339,15215.947457,NaN,0.013279,NaN,NaN,NaN,NaN,202.047882,89.176844,112.871038,4.408210e+10,0.032538,0.040437,NaN,NaN,NaN,9.541572,9.642373,NaN,-0.010454,NaN,NaN,NaN,NaN,-0.100801,-0.052795,-0.048005,0.0,-0.001288,0.001823,NaN,NaN,NaN,1316.173860,1315.024596,NaN,0.000874,NaN,NaN,NaN,NaN,1.149264,0.487255,0.662009,4.0,-0.280786,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.4

In [59]:
# Quick date coverage check
print("Prices date range:", prices["date"].min().date(), "->", prices["date"].max().date())
print("Trends date range:", trends["date"].min().date(), "->", trends["date"].max().date())

common_start = max(prices["date"].min(), trends["date"].min())
common_end = min(prices["date"].max(), trends["date"].max())
print("Common overlap:", common_start.date(), "->", common_end.date())

Prices date range: 2018-01-02 -> 2026-04-16
Trends date range: 2018-01-01 -> 2026-04-14
Common overlap: 2018-01-02 -> 2026-04-14


In [60]:
# Merge on date (inner join keeps only dates present in both)
df_merged = prices.merge(trends, on="date", how="inner")
df_merged = df_merged.sort_values("date").reset_index(drop=True)

for col in df_merged.columns:
    if col != "date":
        df_merged[col] = pd.to_numeric(df_merged[col], errors="coerce")


def safe_series(df: pd.DataFrame, name: str) -> pd.Series:
    if name in df.columns:
        return pd.to_numeric(df[name], errors="coerce").fillna(0.0)
    return pd.Series(0.0, index=df.index, dtype=float)


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """Composite features for tree/linear models (no future leakage — uses same row or lags)."""
    out = df.copy()
    eth = safe_series(out, "ETH")
    btc = safe_series(out, "BTC")

    out["eth_ret_14d"] = eth.pct_change(14)
    out["btc_ret_14d"] = btc.pct_change(14)

    trend_keywords = [
        "bitcoin", "cryptocurrency", "ethereum", "crypto", "btc", "eth",
        "crypto crash", "bitcoin crash", "buy bitcoin", "sell bitcoin",
        "crypto bull run", "solana", "altcoin", "bitcoin price", "crypto wallet",
    ]
    for kw in trend_keywords:
        if kw not in out.columns:
            continue
        s = pd.to_numeric(out[kw], errors="coerce")
        out[f"{kw}_d1"] = s.diff(1)
        rm = s.rolling(7).mean()
        rs = s.rolling(7).std()
        out[f"{kw}_z7"] = (s - rm) / (rs + 1e-9)

    out["sentiment_balance"] = safe_series(out, "eth_news_pos_ratio") - 0.5
    out["news_intensity"] = np.log1p(safe_series(out, "eth_news_count").clip(lower=0))
    out["macro_stress"] = (
        safe_series(out, "treasury_10y_change_7d")
        + safe_series(out, "fed_rate_change_7d")
        + safe_series(out, "cpi_yoy_pct") / 100.0
    )
    out["trend_strength"] = (
        safe_series(out, "eth_macd_hist").abs()
        + safe_series(out, "btc_macd_hist").abs()
        + safe_series(out, "eth_bb_z_20").abs()
    )
    out["risk_regime"] = safe_series(out, "eth_vol_14d") + safe_series(out, "vix_vol_14d")
    out["vol_spread"] = safe_series(out, "eth_vol_14d") - safe_series(out, "btc_vol_14d")
    out["rsi_extreme"] = (safe_series(out, "eth_rsi_14") - 50.0).abs() / 50.0
    out["momentum_gap_7"] = safe_series(out, "eth_ret_7d") - safe_series(out, "btc_ret_7d")
    out["momentum_gap_14"] = out["eth_ret_14d"] - out["btc_ret_14d"]
    out["trend_alignment"] = np.sign(safe_series(out, "eth_ret_7d")) * np.sign(
        safe_series(out, "btc_ret_7d")
    )

    obv_eth = safe_series(out, "eth_obv")
    obv_btc = safe_series(out, "btc_obv")
    for prefix, series in [("eth", obv_eth), ("btc", obv_btc)]:
        ma = series.rolling(20).mean()
        sd = series.rolling(20).std()
        out[f"{prefix}_obv_z20"] = (series - ma) / (sd + 1e-9)

    out["obv_spread"] = safe_series(out, "eth_obv_z20") - safe_series(out, "btc_obv_z20")

    # Extra signals for direction models (same-day + lags only; no future leakage)
    er1 = safe_series(out, "eth_ret_1d")
    br1 = safe_series(out, "btc_ret_1d")
    out["eth_btc_price_ratio"] = eth / (btc + 1e-9)
    out["eth_btc_ratio_ret_1d"] = out["eth_btc_price_ratio"].pct_change(1)
    rm7e, rs7e = er1.rolling(7).mean(), er1.rolling(7).std()
    out["eth_ret_1d_z7"] = (er1 - rm7e) / (rs7e + 1e-9)
    rm7b, rs7b = br1.rolling(7).mean(), br1.rolling(7).std()
    out["btc_ret_1d_z7"] = (br1 - rm7b) / (rs7b + 1e-9)
    ev7 = safe_series(out, "eth_vol_7d")
    out["eth_vol_7d_z7"] = (ev7 - ev7.rolling(7).mean()) / (ev7.rolling(7).std() + 1e-9)
    vx1 = safe_series(out, "vix_ret_1d")
    out["vix_ret_1d_z7"] = (vx1 - vx1.rolling(7).mean()) / (vx1.rolling(7).std() + 1e-9)
    bes = safe_series(out, "btc_eth_spread_1d")
    out["btc_eth_spread_1d_z7"] = (bes - bes.rolling(7).mean()) / (bes.rolling(7).std() + 1e-9)
    st = safe_series(out, "sentiment_balance")
    mg = safe_series(out, "momentum_gap_7")
    for lag in (1, 2, 3):
        out[f"eth_ret_1d_lag{lag}"] = er1.shift(lag)
        out[f"btc_ret_1d_lag{lag}"] = br1.shift(lag)
        out[f"sentiment_balance_lag{lag}"] = st.shift(lag)
        out[f"momentum_gap_7_lag{lag}"] = mg.shift(lag)
    if "date" in out.columns:
        dow = pd.to_datetime(out["date"]).dt.dayofweek.astype(float)
        out["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
        out["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    out["eth_btc_ret_corr_30"] = er1.rolling(30, min_periods=10).corr(br1)
    out["vix_x_eth_vol"] = safe_series(out, "VIX") * safe_series(out, "eth_vol_14d")

    if "buy bitcoin_z7" in out.columns:
        out["buy_signal_pressure"] = out["buy bitcoin_z7"]
    if "altcoin_z7" in out.columns:
        out["altcoin_pressure"] = out["altcoin_z7"]

    return out


df_ml = add_engineered_features(df_merged)

print("Merged (before features) shape:", df_merged.shape)
print("Final ML table shape:", df_ml.shape)
print("Missing values total:", int(df_ml.isna().sum().sum()))
print("Column count:", len(df_ml.columns))
df_ml.head()

Merged (before features) shape: (2081, 101)
Final ML table shape: (2081, 171)
Missing values total: 1163
Column count: 171


,date,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,eth_ema_12,eth_ema_26,eth_sma_ratio_10,eth_ema_ratio_12_26,eth_vol_7d,eth_vol_14d,eth_bb_z_20,eth_rsi_14,eth_macd,eth_macd_signal,eth_macd_hist,eth_obv,btc_ret_1d,btc_ret_3d,btc_ret_7d,btc_sma_10,btc_sma_30,btc_ema_12,btc_ema_26,btc_sma_ratio_10,btc_ema_ratio_12_26,btc_vol_7d,btc_vol_14d,btc_bb_z_20,btc_rsi_14,btc_macd,btc_macd_signal,btc_macd_hist,btc_obv,vix_ret_1d,vix_ret_3d,vix_ret_7d,vix_sma_10,vix_sma_30,vix_ema_12,vix_ema_26,vix_sma_ratio_10,vix_ema_ratio_12_26,vix_vol_7d,vix_vol_14d,vix_bb_z_20,vix_rsi_14,vix_macd,vix_macd_signal,vix_macd_hist,vix_obv,gc=f_ret_1d,gc=f_ret_3d,gc=f_ret_7d,gc=f_sma_10,gc=f_sma_30,gc=f_ema_12,gc=f_ema_26,gc=f_sma_ratio_10,gc=f_ema_ratio_12_26,gc=f_vol_7d,gc=f_vol_14d,gc=f_bb_z_20,gc=f_rsi_14,gc=f_macd,gc=f_macd_signal,gc=f_macd_hist,gc=f_obv,btc_eth_spread_1d,eth_news_sent_mean,eth_news_sent_wmean,eth_news_sent_std,eth_news_count,eth_news_pos_ratio,eth_news_neg_ratio,federal_funds_rate,cpi,treasury_10y,fed_rate_change_7d,treasury_10y_change_7d,cpi_yoy_pct,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet,eth_ret_14d,btc_ret_14d,bitcoin_d1,bitcoin_z7,cryptocurrency_d1,cryptocurrency_z7,ethereum_d1,ethereum_z7,crypto_d1,crypto_z7,btc_d1,btc_z7,eth_d1,eth_z7,crypto crash_d1,crypto crash_z7,bitcoin crash_d1,bitcoin crash_z7,buy bitcoin_d1,buy bitcoin_z7,sell bitcoin_d1,sell bitcoin_z7,crypto bull run_d1,crypto bull run_z7,solana_d1,solana_z7,altcoin_d1,altcoin_z7,bitcoin price_d1,bitcoin price_z7,crypto wallet_d1,crypto wallet_z7,sentiment_balance,news_intensity,macro_stress,trend_strength,risk_regime,vol_spread,rsi_extreme,momentum_gap_7,momentum_gap_14,trend_alignment,eth_obv_z20,btc_obv_z20,obv_spread,eth_btc_price_ratio,eth_btc_ratio_ret_1d,eth_ret_1d_z7,btc_ret_1d_z7,eth_vol_7d_z7,vix_ret_1d_z7,btc_eth_spread_1d_z7,eth_ret_1d_lag1,btc_ret_1d_lag1,sentiment_balance_lag1,momentum_gap_7_lag1,eth_ret_1d_lag2,btc_ret_1d_lag2,sentiment_balance_lag2,momentum_gap_7_lag2,eth_ret_1d_lag3,btc_ret_1d_lag3,sentiment_balance_lag3,momentum_gap_7_lag3,dow_sin,dow_cos,eth_btc_ret_corr_30,vix_x_eth_vol,buy_signal_pressure,altcoin_pressure
0,2018-01-02,1313.699951,9.77,884.443970,14982.099609,NaN,NaN,NaN,NaN,NaN,884.443970,884.443970,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,14982.099609,14982.099609,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,9.770000,9.770000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,1313.699951,1313.699951,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.46,NaN,NaN,NaN,53,10,8,8,12,69,0,3,42,6,0,1,2,48,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.5,0.000000,0.0,0.000000,0.0,0.0,1.0,0.0,NaN,0.0,NaN,NaN,0.0,0.059033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.781831,0.623490,NaN,0.0,NaN,NaN
1,2018-01-03,1316.199951,9.15,962.719971,15201.000000,0.088503,NaN,NaN,NaN,NaN,896.486431,890.242192,NaN,0.007014,NaN,NaN,NaN,NaN,6.244239,1.248848,4.995392,5.093160e+09,0.014611,NaN,NaN,NaN,NaN,15015.776593,14998.314453,NaN,0.001164,NaN,NaN,NaN,NaN,17.462139,3.492428,13.969712,1.687190e+10,-0.063460,NaN,NaN,NaN,NaN,9.674616,9.724074,NaN,-0.005086,NaN,NaN,NaN,NaN,-0.049459,-0.009892,-0.039567,0.0,0.001903,NaN,NaN,NaN,NaN,1314.084567,1313.885136,NaN,0.000152,NaN,NaN,NaN,NaN,0.199430,0.039886,0.159544,42.0,-0.073892,0.03173,0.03173,0.0,1.0,0.0,0.0,1.42,247.867,2.44,NaN,NaN,NaN,53,12,8,10,12,73,1,3,51,7,0,1,3,46,1,NaN,NaN,0.0,NaN,2.0,NaN,0.0,NaN,2.0,NaN,0.0,NaN,4.0,NaN,1.0,NaN,0.0,NaN,9.0,NaN,1.0,NaN,0.0,NaN,0.0,NaN,1.0,NaN,-2.0,NaN,0.0,NaN,-0.5,0.693147,0.0,18.965103,0.0,0.0,1.0,0.0,NaN,0.0,NaN,NaN,0.0,0.063333,0.07282

In [61]:
# Final validation
print("Date range:", df_ml["date"].min().date(), "->", df_ml["date"].max().date())
print("Duplicate dates:", int(df_ml["date"].duplicated().sum()))
print("Rows:", len(df_ml))
print("Columns:", len(df_ml.columns))

df_ml.tail()

Date range: 2018-01-02 -> 2026-04-14
Duplicate dates: 0
Rows: 2081
Columns: 171


,date,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,eth_ema_12,eth_ema_26,eth_sma_ratio_10,eth_ema_ratio_12_26,eth_vol_7d,eth_vol_14d,eth_bb_z_20,eth_rsi_14,eth_macd,eth_macd_signal,eth_macd_hist,eth_obv,btc_ret_1d,btc_ret_3d,btc_ret_7d,btc_sma_10,btc_sma_30,btc_ema_12,btc_ema_26,btc_sma_ratio_10,btc_ema_ratio_12_26,btc_vol_7d,btc_vol_14d,btc_bb_z_20,btc_rsi_14,btc_macd,btc_macd_signal,btc_macd_hist,btc_obv,vix_ret_1d,vix_ret_3d,vix_ret_7d,vix_sma_10,vix_sma_30,vix_ema_12,vix_ema_26,vix_sma_ratio_10,vix_ema_ratio_12_26,vix_vol_7d,vix_vol_14d,vix_bb_z_20,vix_rsi_14,vix_macd,vix_macd_signal,vix_macd_hist,vix_obv,gc=f_ret_1d,gc=f_ret_3d,gc=f_ret_7d,gc=f_sma_10,gc=f_sma_30,gc=f_ema_12,gc=f_ema_26,gc=f_sma_ratio_10,gc=f_ema_ratio_12_26,gc=f_vol_7d,gc=f_vol_14d,gc=f_bb_z_20,gc=f_rsi_14,gc=f_macd,gc=f_macd_signal,gc=f_macd_hist,gc=f_obv,btc_eth_spread_1d,eth_news_sent_mean,eth_news_sent_wmean,eth_news_sent_std,eth_news_count,eth_news_pos_ratio,eth_news_neg_ratio,federal_funds_rate,cpi,treasury_10y,fed_rate_change_7d,treasury_10y_change_7d,cpi_yoy_pct,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet,eth_ret_14d,btc_ret_14d,bitcoin_d1,bitcoin_z7,cryptocurrency_d1,cryptocurrency_z7,ethereum_d1,ethereum_z7,crypto_d1,crypto_z7,btc_d1,btc_z7,eth_d1,eth_z7,crypto crash_d1,crypto crash_z7,bitcoin crash_d1,bitcoin crash_z7,buy bitcoin_d1,buy bitcoin_z7,sell bitcoin_d1,sell bitcoin_z7,crypto bull run_d1,crypto bull run_z7,solana_d1,solana_z7,altcoin_d1,altcoin_z7,bitcoin price_d1,bitcoin price_z7,crypto wallet_d1,crypto wallet_z7,sentiment_balance,news_intensity,macro_stress,trend_strength,risk_regime,vol_spread,rsi_extreme,momentum_gap_7,momentum_gap_14,trend_alignment,eth_obv_z20,btc_obv_z20,obv_spread,eth_btc_price_ratio,eth_btc_ratio_ret_1d,eth_ret_1d_z7,btc_ret_1d_z7,eth_vol_7d_z7,vix_ret_1d_z7,btc_eth_spread_1d_z7,eth_ret_1d_lag1,btc_ret_1d_lag1,sentiment_balance_lag1,momentum_gap_7_lag1,eth_ret_1d_lag2,btc_ret_1d_lag2,sentiment_balance_lag2,momentum_gap_7_lag2,eth_ret_1d_lag3,btc_ret_1d_lag3,sentiment_balance_lag3,momentum_gap_7_lag3,dow_sin,dow_cos,eth_btc_ret_corr_30,vix_x_eth_vol,buy_signal_pressure,altcoin_pressure
2076,2026-04-08,4749.500000,21.040001,2190.335449,71123.359375,-0.022960,0.064897,0.099968,2108.258813,2099.789148,2131.895933,2136.921564,0.038931,-0.002352,0.035040,0.032045,0.567288,49.098139,-5.025630,-21.877817,16.852186,7.770804e+11,-0.011361,0.063311,0.072130,68825.565625,69673.406510,69519.742298,70080.041560,0.033386,-0.007995,0.022883,0.023215,0.465726,49.652877,-560.299262,-910.716172,350.416910,6.252164e+11,-0.183863,-0.118559,-0.322383,25.908,24.631667,25.012469,24.532641,-0.187896,0.019559,0.094306,0.093505,-1.879550,42.314993,0.479828,1.218178,-0.738350,0.0,0.019841,0.021068,0.057324,4608.900000,4865.069987,4671.829274,4751.479761,0.030506,-0.016763,0.019652,0.029032,0.143533,45.167956,-79.650486,-99.389279,19.738793,1519690.0,0.011598,0.0,0.0,0.0,0.0,0.0,0.0,3.64,330.213,4.29,0.0,-0.15,0.04609,22,1,3,9,9,36,0,0,7,1,0,5,0,18,1,-0.006052,-0.001715,2.0,-0.129010,0.0,-0.585540,1.0,0.000000,0.0,-0.561769,1.0,0.944911,5.0,0.112169,-1.0,-1.463850,-1.0,-1.069045,0.0,-0.517549,0.0,-0.377964,0.0,0.0,0.0,-0.705476,0.0,0.0,4.0,0.129500,0.0,0.0,-0.5,0.0,-0.149539,367.836384,0.125550,0.008830,0.018037,0.027838,-0.004336,1.0,1.204356,-0.122128,1.326483,0.030796,-0.011732,-1.061245,-0.943165,0.577716,-1.420396,0.961550,0.063596,0.044741,-0.5,0.042701,0.024751,0.029471,-0.5,0.006563,-0.038286,-0.017480,-0.5,0.005610,0.974928,-0.222521,0.950356,0.674230,-0.517549,0.0
2077,2026-04-09,4792.200195,19.490000,2189.144043,71767.828125,-0.000544,0.038611,0.081852,2110.371069,2104.273018,2140.703335,2140.789896,0.037327,-0.000040,0.035452,0.030861,0.488448,53.835028,-0.086561,-17.519565,17.433005,7.598618e+11,0.009061,0.042231,0.076117,68871.360156,69800.329948,69865.601656,70205.062787,0.042056,-0.

In [62]:
# Save final ML datasets (same table: base filename + *_features for selective / GBDT scripts)
df_ml.to_csv(OUTPUT_PATH, index=False)
df_ml.to_csv(OUTPUT_FEATURES_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {OUTPUT_FEATURES_PATH}")

Saved: eth_ml_dataset.csv
Saved: eth_ml_dataset_features.csv


In [63]:
df_ml

,date,GC=F,VIX,ETH,BTC,eth_ret_1d,eth_ret_3d,eth_ret_7d,eth_sma_10,eth_sma_30,eth_ema_12,eth_ema_26,eth_sma_ratio_10,eth_ema_ratio_12_26,eth_vol_7d,eth_vol_14d,eth_bb_z_20,eth_rsi_14,eth_macd,eth_macd_signal,eth_macd_hist,eth_obv,btc_ret_1d,btc_ret_3d,btc_ret_7d,btc_sma_10,btc_sma_30,btc_ema_12,btc_ema_26,btc_sma_ratio_10,btc_ema_ratio_12_26,btc_vol_7d,btc_vol_14d,btc_bb_z_20,btc_rsi_14,btc_macd,btc_macd_signal,btc_macd_hist,btc_obv,vix_ret_1d,vix_ret_3d,vix_ret_7d,vix_sma_10,vix_sma_30,vix_ema_12,vix_ema_26,vix_sma_ratio_10,vix_ema_ratio_12_26,vix_vol_7d,vix_vol_14d,vix_bb_z_20,vix_rsi_14,vix_macd,vix_macd_signal,vix_macd_hist,vix_obv,gc=f_ret_1d,gc=f_ret_3d,gc=f_ret_7d,gc=f_sma_10,gc=f_sma_30,gc=f_ema_12,gc=f_ema_26,gc=f_sma_ratio_10,gc=f_ema_ratio_12_26,gc=f_vol_7d,gc=f_vol_14d,gc=f_bb_z_20,gc=f_rsi_14,gc=f_macd,gc=f_macd_signal,gc=f_macd_hist,gc=f_obv,btc_eth_spread_1d,eth_news_sent_mean,eth_news_sent_wmean,eth_news_sent_std,eth_news_count,eth_news_pos_ratio,eth_news_neg_ratio,federal_funds_rate,cpi,treasury_10y,fed_rate_change_7d,treasury_10y_change_7d,cpi_yoy_pct,bitcoin,cryptocurrency,ethereum,crypto,btc,eth,crypto crash,bitcoin crash,buy bitcoin,sell bitcoin,crypto bull run,solana,altcoin,bitcoin price,crypto wallet,eth_ret_14d,btc_ret_14d,bitcoin_d1,bitcoin_z7,cryptocurrency_d1,cryptocurrency_z7,ethereum_d1,ethereum_z7,crypto_d1,crypto_z7,btc_d1,btc_z7,eth_d1,eth_z7,crypto crash_d1,crypto crash_z7,bitcoin crash_d1,bitcoin crash_z7,buy bitcoin_d1,buy bitcoin_z7,sell bitcoin_d1,sell bitcoin_z7,crypto bull run_d1,crypto bull run_z7,solana_d1,solana_z7,altcoin_d1,altcoin_z7,bitcoin price_d1,bitcoin price_z7,crypto wallet_d1,crypto wallet_z7,sentiment_balance,news_intensity,macro_stress,trend_strength,risk_regime,vol_spread,rsi_extreme,momentum_gap_7,momentum_gap_14,trend_alignment,eth_obv_z20,btc_obv_z20,obv_spread,eth_btc_price_ratio,eth_btc_ratio_ret_1d,eth_ret_1d_z7,btc_ret_1d_z7,eth_vol_7d_z7,vix_ret_1d_z7,btc_eth_spread_1d_z7,eth_ret_1d_lag1,btc_ret_1d_lag1,sentiment_balance_lag1,momentum_gap_7_lag1,eth_ret_1d_lag2,btc_ret_1d_lag2,sentiment_balance_lag2,momentum_gap_7_lag2,eth_ret_1d_lag3,btc_ret_1d_lag3,sentiment_balance_lag3,momentum_gap_7_lag3,dow_sin,dow_cos,eth_btc_ret_corr_30,vix_x_eth_vol,buy_signal_pressure,altcoin_pressure
0,2018-01-02,1313.699951,9.770000,884.443970,14982.099609,NaN,NaN,NaN,NaN,NaN,884.443970,884.443970,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,14982.099609,14982.099609,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000e+00,NaN,NaN,NaN,NaN,NaN,9.770000,9.770000,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,NaN,NaN,NaN,NaN,1313.699951,1313.699951,NaN,0.000000,NaN,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.0,NaN,0.00000,0.00000,0.0,0.0,0.0,0.0,1.42,247.867,2.46,NaN,NaN,NaN,53,10,8,8,12,69,0,3,42,6,0,1,2,48,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.5,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,NaN,0.0,NaN,NaN,0.000000,0.059033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.781831,0.623490,NaN,0.000000,NaN,NaN
1,2018-01-03,1316.199951,9.150000,962.719971,15201.000000,0.088503,NaN,NaN,NaN,NaN,896.486431,890.242192,NaN,0.007014,NaN,NaN,NaN,NaN,6.244239,1.248848,4.995392,5.093160e+09,0.014611,NaN,NaN,NaN,NaN,15015.776593,14998.314453,NaN,0.001164,NaN,NaN,NaN,NaN,17.462139,3.492428,13.969712,1.687190e+10,-0.063460,NaN,NaN,NaN,NaN,9.674616,9.724074,NaN,-0.005086,NaN,NaN,NaN,NaN,-0.049459,-0.009892,-0.039567,0.0,0.001903,NaN,NaN,NaN,NaN,1314.084567,1313.885136,NaN,0.000152,NaN,NaN,NaN,NaN,0.199430,0.039886,0.159544,42.0,-0.073892,0.03173,0.03173,0.0,1.0,0.0,0.0,1.42,247.867,2.44,NaN,NaN,NaN,53,12,8,10,12,73,1,3,51,7,0,1,3,46,1,NaN,NaN,0.0,NaN,2.0,NaN,0.0,NaN,2.0,NaN,0.0,NaN,4.0,NaN,1.0,NaN,0.0,NaN,9.0,NaN,1.0,NaN,0.0,NaN,0.0,NaN,1.0,NaN,-2.0,NaN,0.0,NaN,-0.5,0.693147,0.000000,18.965103,0.00